# 🦒 Fine-tuning Cellpose with BioEngine ⚙️☁️

Welcome to this tutorial on fine-tuning the Cellpose cell segmentation model using BioEngine!

## 🔍 What You'll Learn

In this guide, you'll discover how to:
1. 🔌 Connect to a Cellpose Fine-tuning service on BioEngine
2. 🚀 Start a new training session on annotated brightfield microscopy images
3. 📊 Visualize training and test losses after each session
4. 🔄 Continue training to accumulate more epochs
5. 📦 Export the trained model as a BioImage.IO-compatible artifact
6. 🗂️ List models trained on a given dataset

## 🧪 How It Works

BioEngine's Cellpose Fine-tuning service lets you train Cellpose on your own annotated data stored in the Hypha artifact manager:

1. 📂 Your images and masks are stored in a Hypha artifact
2. 🏋️ The service downloads the data and runs GPU-accelerated training on the server
3. 📈 You poll for progress and inspect per-epoch metrics
4. 💾 When satisfied, you export the model as a BioImage.IO package

## 📚 Dataset Used in This Tutorial

We use the artifact **`bioimage-io/annotation-mnw8359y-ehab`** — a collection of annotated brightfield microscopy images gathered with the [BioImage.IO Colab](https://bioimage.io/#/colab) collaborative annotation platform. The artifact contains:

| Folder | Contents |
|--------|----------|
| `train_images/` | 13 brightfield PNG images for training |
| `test_images/` | 2 brightfield PNG images for validation |
| `masks_cells/` | Instance-segmentation PNG masks (and GeoJSON files) for all 15 images |

The service automatically matches each image to its corresponding mask by filename stem.

## ⚠️ Important Note

This tutorial uses the public Hypha server at [https://hypha.aicell.io](https://hypha.aicell.io) and the shared service **`bioimage-io/cellpose-finetuning`**. This service is provided by the AI4Life team for evaluation purposes and may change without notice.

For production use or sensitive data, deploy your own BioEngine worker following the [Deployment Guidelines](https://github.com/aicell-lab/bioengine-worker).


## 🧩 Setup and Installation

In [ ]:
try:
    # For pyodide in the browser
    import micropip

    await micropip.install(
        ["pyodide-http", "hypha-rpc", "matplotlib", "numpy"]
    )

    import pyodide_http

    pyodide_http.patch_all()
except ImportError:
    # For native Python with pip
    import subprocess

    subprocess.call(["pip", "install", "hypha-rpc", "matplotlib", "numpy"])

import asyncio
import datetime
import json
import os

import matplotlib.pyplot as plt
import numpy as np
from hypha_rpc import connect_to_server, login

## 🔌☁️ Connect to the Hypha Server

We connect to [https://hypha.aicell.io](https://hypha.aicell.io). A browser login will open if `HYPHA_TOKEN` is not set in your environment.

In [ ]:
SERVER_URL = "https://hypha.aicell.io"

In [ ]:
token = os.getenv("HYPHA_TOKEN") or await login({"server_url": SERVER_URL})
server = await connect_to_server(
    {"server_url": SERVER_URL, "token": token, "method_timeout": 300}
)

print(f"✅ Connected to workspace: {server.config.workspace}")

## 🧬 Connect to the Cellpose Fine-tuning Service

The public Cellpose Fine-tuning service is available at `bioimage-io/cellpose-finetuning`.

In [ ]:
CELLPOSE_SERVICE_ID = "bioimage-io/cellpose-finetuning"

cellpose_svc = await server.get_service(CELLPOSE_SERVICE_ID)

print(f"✅ Connected to service: {CELLPOSE_SERVICE_ID}")

## 📂 Dataset Configuration

We use a pre-existing Hypha artifact with annotated brightfield images collected through the BioImage.IO Colab annotation platform.

The artifact structure is:
```
bioimage-io/annotation-mnw8359y-ehab/
├── train_images/      ← 13 brightfield PNG images
├── test_images/       ← 2 brightfield PNG images
└── masks_cells/       ← PNG instance-segmentation masks for all 15 images
                         (+ GeoJSON files — excluded via *.png pattern)
```

We use the `*.png` glob pattern to select only the PNG masks from `masks_cells/`, avoiding the GeoJSON annotation files. The service automatically matches each image to its mask by filename stem.

In [ ]:
DATASET_ARTIFACT  = "bioimage-io/annotation-mnw8359y-ehab"

TRAIN_IMAGES      = "train_images/*.png"
TRAIN_ANNOTATIONS = "masks_cells/*.png"
TEST_IMAGES       = "test_images/*.png"
TEST_ANNOTATIONS  = "masks_cells/*.png"

print(f"Dataset artifact : {DATASET_ARTIFACT}")
print(f"Train images     : {TRAIN_IMAGES}")
print(f"Train masks      : {TRAIN_ANNOTATIONS}")
print(f"Test images      : {TEST_IMAGES}")
print(f"Test masks       : {TEST_ANNOTATIONS}")

---

## 🏋️ Step 1: Start a Training Session (10 Epochs)

We fine-tune the base **CellposeSAM (`cpsam`)** model for **10 epochs** with per-epoch validation (`validation_interval=1`) so we can inspect the loss curve in detail.

`start_training` launches training asynchronously and immediately returns a `session_id`. We then poll `get_training_status` until the run finishes.

In [ ]:
session_result = await cellpose_svc.start_training(
    artifact=DATASET_ARTIFACT,
    train_images=TRAIN_IMAGES,
    train_annotations=TRAIN_ANNOTATIONS,
    test_images=TEST_IMAGES,
    test_annotations=TEST_ANNOTATIONS,
    model="cpsam",
    n_epochs=10,
    validation_interval=1,
)

session_id = session_result["session_id"]

print("🚀 Training started!")
print(json.dumps(session_result, indent=2, default=str))

### ⏳ Wait for Training to Complete

Poll `get_training_status` every 15 seconds until the session reaches a terminal state.

In [ ]:
TERMINAL_STATES = {"completed", "failed", "stopped"}

prev_epoch = -1
while True:
    status = await cellpose_svc.get_training_status(session_id=session_id)
    st      = status.get("status_type", "unknown")
    epoch   = status.get("current_epoch") or 0
    total   = status.get("total_epochs") or "?"
    n_train = status.get("n_train") or "?"
    n_test  = status.get("n_test") or "?"
    elapsed = status.get("elapsed_seconds")
    msg     = status.get("message", "")

    if epoch != prev_epoch or st in TERMINAL_STATES:
        elapsed_str = f"{elapsed:.0f}s" if elapsed is not None else "—"
        print(
            f"[{st:12s}] epoch {epoch}/{total}  "
            f"train={n_train}  test={n_test}  elapsed={elapsed_str}  {msg}"
        )
        prev_epoch = epoch

    if st in TERMINAL_STATES:
        break

    await asyncio.sleep(15)

print(f"\n✅ Session '{session_id}' finished with status: {status['status_type']}")

---

## 📊 Step 2: Visualise Training and Validation Metrics

After the training run we inspect:
- **Train loss** per epoch
- **Test (validation) loss** per epoch
- **Pixel accuracy** per epoch (from `test_metrics`)

In [ ]:
status = await cellpose_svc.get_training_status(session_id=session_id)

train_losses = status.get("train_losses") or []
test_losses  = status.get("test_losses")  or []
test_metrics = status.get("test_metrics") or []
n_epochs_run = len(train_losses)
epochs       = list(range(1, n_epochs_run + 1))

print(f"Epochs completed : {n_epochs_run}")
print(f"Training samples : {status.get('n_train')}")
print(f"Test samples     : {status.get('n_test')}")
print(f"Elapsed time     : {status.get('elapsed_seconds', 0):.1f}s")
print()
print("Per-epoch summary:")
for i, (tl, vl) in enumerate(zip(train_losses, test_losses)):
    acc = test_metrics[i]["pixel_accuracy"] if i < len(test_metrics) and test_metrics[i] else None
    acc_str = f"{acc:.4f}" if acc is not None else "—"
    print(f"  Epoch {i+1:2d}  train_loss={tl:.4f}  test_loss={vl:.4f}  pixel_acc={acc_str}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(epochs, train_losses, marker="o", label="Train loss", color="steelblue")
if test_losses:
    ax.plot(epochs, test_losses, marker="s", linestyle="--", label="Test loss", color="coral")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training & Test Loss (Session 1, 10 Epochs)")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
if test_metrics:
    acc_values   = [m["pixel_accuracy"] if m else None for m in test_metrics]
    valid_epochs = [e for e, a in zip(epochs, acc_values) if a is not None]
    valid_acc    = [a for a in acc_values if a is not None]
    ax.plot(valid_epochs, valid_acc, marker="^", color="seagreen", label="Pixel accuracy")
    ax.set_ylim(0, 1)
ax.set_xlabel("Epoch")
ax.set_ylabel("Pixel Accuracy")
ax.set_title("Validation Pixel Accuracy (Session 1)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 🗂️ Step 3: List Training Sessions for This Dataset

`list_training_sessions` returns all sessions the server knows about, optionally filtered by dataset artifact ID. This is useful to find earlier sessions or check on sessions from previous notebook runs.

In [ ]:
sessions = await cellpose_svc.list_training_sessions(
    dataset_artifact_ids=[DATASET_ARTIFACT]
)

print(f"Found {len(sessions)} training session(s) for dataset '{DATASET_ARTIFACT}':\n")

for sid, s in sessions.items():
    st      = s.get("status_type", "unknown")
    epoch   = s.get("current_epoch") or 0
    total   = s.get("total_epochs") or "?"
    model   = s.get("model", "?")
    elapsed = s.get("elapsed_seconds")
    elapsed_str = f"{elapsed:.0f}s" if elapsed is not None else "—"
    exported = s.get("exported_artifact_id") or "—"
    print(
        f"  Session : {sid}\n"
        f"    Status    : {st}  |  Epochs: {epoch}/{total}  |  Elapsed: {elapsed_str}\n"
        f"    Base model: {model}  |  Exported: {exported}\n"
    )

assert session_id in sessions, "⚠️  Our session was not found in the list!"
print(f"✅ Session '{session_id}' is present in the list.")

---

## 🔄 Step 4: Continue Training for 10 More Epochs

`restart_training` creates a new session that picks up from the checkpoint of an existing **completed** session. The weights from the previous run become the starting model, accumulating training across sessions.

Here we add 10 more epochs, bringing the effective total to **20 epochs**.

In [ ]:
continued_result = await cellpose_svc.restart_training(
    session_id=session_id,
    n_epochs=10,
)

continued_session_id = continued_result["session_id"]

print("🔄 Continued training started!")
print(json.dumps(continued_result, indent=2, default=str))

In [ ]:
prev_epoch = -1
while True:
    status2 = await cellpose_svc.get_training_status(session_id=continued_session_id)
    st2     = status2.get("status_type", "unknown")
    epoch2  = status2.get("current_epoch") or 0
    total2  = status2.get("total_epochs") or "?"
    elapsed2 = status2.get("elapsed_seconds")
    msg2    = status2.get("message", "")

    if epoch2 != prev_epoch or st2 in TERMINAL_STATES:
        elapsed_str = f"{elapsed2:.0f}s" if elapsed2 is not None else "—"
        print(f"[{st2:12s}] epoch {epoch2}/{total2}  elapsed={elapsed_str}  {msg2}")
        prev_epoch = epoch2

    if st2 in TERMINAL_STATES:
        break

    await asyncio.sleep(15)

print(f"\n✅ Continued session '{continued_session_id}' finished with status: {status2['status_type']}")

---

## 📊 Step 5: Visualise Losses and Metrics for Both Sessions

We plot the losses from **both** sessions as a combined 20-epoch view, with a vertical line marking the session boundary.

In [ ]:
status2 = await cellpose_svc.get_training_status(session_id=continued_session_id)

train_losses2 = status2.get("train_losses") or []
test_losses2  = status2.get("test_losses")  or []
test_metrics2 = status2.get("test_metrics") or []
n2 = len(train_losses2)

all_train   = train_losses  + train_losses2
all_test    = test_losses   + test_losses2
all_metrics = test_metrics  + test_metrics2
all_epochs  = list(range(1, len(all_train) + 1))

print(f"Session 1: {n_epochs_run} epochs  (base model: cpsam)")
print(f"Session 2: {n2} epochs  (started from Session 1 checkpoint)")
print(f"Total effective epochs: {len(all_epochs)}")
print()
print("Session 2 per-epoch summary:")
for i, (tl, vl) in enumerate(zip(train_losses2, test_losses2)):
    acc = test_metrics2[i]["pixel_accuracy"] if i < len(test_metrics2) and test_metrics2[i] else None
    acc_str = f"{acc:.4f}" if acc is not None else "—"
    print(f"  Epoch {n_epochs_run + i + 1:2d}  train_loss={tl:.4f}  test_loss={vl:.4f}  pixel_acc={acc_str}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(all_epochs[:n_epochs_run], all_train[:n_epochs_run],
        marker="o", color="steelblue", label="Train loss (session 1)")
ax.plot(all_epochs[n_epochs_run:], all_train[n_epochs_run:],
        marker="o", color="navy", label="Train loss (session 2)")
ax.plot(all_epochs[:n_epochs_run], all_test[:n_epochs_run],
        marker="s", linestyle="--", color="coral", label="Test loss (session 1)")
ax.plot(all_epochs[n_epochs_run:], all_test[n_epochs_run:],
        marker="s", linestyle="--", color="darkred", label="Test loss (session 2)")
ax.axvline(x=n_epochs_run + 0.5, color="gray", linestyle=":", label="Session boundary")
ax.set_xlabel("Cumulative Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training & Test Loss (20 Epochs Total)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
acc_all   = [m["pixel_accuracy"] if m else None for m in all_metrics]
valid_ep  = [e for e, a in zip(all_epochs, acc_all) if a is not None]
valid_acc = [a for a in acc_all if a is not None]
ax.plot(valid_ep, valid_acc, marker="^", color="seagreen", label="Pixel accuracy")
ax.axvline(x=n_epochs_run + 0.5, color="gray", linestyle=":", label="Session boundary")
ax.set_ylim(0, 1)
ax.set_xlabel("Cumulative Epoch")
ax.set_ylabel("Pixel Accuracy")
ax.set_title("Validation Pixel Accuracy (20 Epochs Total)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 📦 Step 6: Export the Model

`export_model` packages the trained weights into a [BioImage.IO](https://bioimage.io)-compatible artifact including model weights, architecture code, test samples, a cover image, and a full RDF descriptor. The artifact is uploaded to the `bioimage-io/colab-annotations` collection by default.

We set **Nils Mechtel from KTH** as the author.

In [ ]:
export_result = await cellpose_svc.export_model(
    session_id=continued_session_id,
    authors=[
        {
            "name": "Nils Mechtel",
            "affiliation": "KTH Royal Institute of Technology",
        }
    ],
)

exported_artifact_id = export_result["artifact_id"]

print("✅ Model exported successfully!")
print(json.dumps(export_result, indent=2, default=str))

The exported artifact contains:

| File | Description |
|------|-------------|
| `model_weights.pth` | Fine-tuned PyTorch model weights |
| `model.py` | Cellpose model architecture adapter |
| `input_sample.npy` / `output_sample.npy` | Example tensors for the BioImage.IO test suite |
| `cover.png` | Auto-generated cover image |
| `rdf.yaml` | BioImage.IO Resource Description File |
| `training_history.json` | Full per-epoch metrics |
| `training_params.json` | Original training configuration |

---

## 🗂️ Step 7: List Exported Models for This Dataset

`list_models_by_dataset` queries the artifact collection and returns all model artifacts trained on the specified dataset. Our newly exported model should appear here.

In [ ]:
models = await cellpose_svc.list_models_by_dataset(
    dataset_id=DATASET_ARTIFACT
)

print(f"Found {len(models)} exported model(s) trained on '{DATASET_ARTIFACT}':\n")

for model in models:
    ts = model.get("created_at")
    created_str = (
        datetime.datetime.fromtimestamp(ts).strftime("%Y-%m-%d %H:%M")
        if ts else "—"
    )
    print(
        f"  ID      : {model.get('id')}\n"
        f"  Name    : {model.get('name')}\n"
        f"  Created : {created_str}\n"
        f"  URL     : {model.get('url')}\n"
    )

model_ids = [m.get("id") for m in models]
assert exported_artifact_id in model_ids, "⚠️  Exported model not found in list!"
print(f"✅ Our exported model '{exported_artifact_id}' is listed.")

---

## 🎉 Summary

In this tutorial you completed a full Cellpose fine-tuning workflow:

| Step | Action | Key API Call |
|------|--------|-------------|
| 1 | Started a 10-epoch training run on annotated brightfield images | `start_training()` |
| 2 | Visualised per-epoch train/test loss and pixel accuracy | `get_training_status()` + matplotlib |
| 3 | Listed all training sessions for the dataset | `list_training_sessions(dataset_artifact_ids=[...])` |
| 4 | Continued training for 10 more epochs from the checkpoint | `restart_training(session_id, n_epochs=10)` |
| 5 | Visualised the combined 20-epoch curves | `get_training_status()` + matplotlib |
| 6 | Exported the final model as a BioImage.IO artifact | `export_model(session_id, authors=[...])` |
| 7 | Listed exported models for the dataset | `list_models_by_dataset(dataset_id=...)` |

### 🔗 Next Steps

- Browse the exported model at the URL printed in Step 6
- Use the [BioEngine Model Runner tutorial](../model-runner/tutorial_model_runner.ipynb) to run inference with your new model
- Collect more annotations via [BioImage.IO Colab](https://bioimage.io/#/colab) and repeat the fine-tuning loop
- Deploy your own BioEngine worker for private data — see the [deployment guide](https://github.com/aicell-lab/bioengine-worker)
